# 04 — Ontology-based trajectory analysis (50-subject subset)

This notebook performs a first longitudinal trajectory analysis on a curated PPMI subset, using the same variable selection and semantic domains defined in the previous steps.

**Input**:
- `data/ppmi_50_subjects.tsv`
- `mapping/ppmi_pdon_pmdo_mapping.csv`

**Primary trajectory target**:
- `updrs3_score` (motor progression)

**Fixed visit design**:
- `BL`, `V04`, `V06`, `V08`, `V10`

**Main outputs**:
- trajectory cohort summary
- subject-level progression table
- progression groups (slow / intermediate / faster)
- domain-level descriptive summaries
- figures saved to `output/trajectory_analysis/`


In [ ]:
!pip -q install pandas numpy matplotlib rdflib

## 1) Project root (Drive recommended)

In [ ]:
from pathlib import Path
import os

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/ppmi-ontology-alignment')
else:
    PROJECT_DIR = Path('/content/ppmi-ontology-alignment')

DATA_DIR = PROJECT_DIR / 'data'
MAP_DIR = PROJECT_DIR / 'mapping'
OUT_DIR = PROJECT_DIR / 'output'
TRAJ_OUT_DIR = OUT_DIR / 'trajectory_analysis'
FIG_DIR = TRAJ_OUT_DIR / 'figures'
for p in [DATA_DIR, MAP_DIR, OUT_DIR, TRAJ_OUT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / 'ppmi_50_subjects.tsv'
MAP_PATH  = MAP_DIR / 'ppmi_pdon_pmdo_mapping.csv'

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_PATH:', DATA_PATH, 'exists=', DATA_PATH.exists())
print('MAP_PATH :', MAP_PATH, 'exists=', MAP_PATH.exists())
print('TRAJ_OUT_DIR:', TRAJ_OUT_DIR)

## 2) Load mapping and analysis subset

In [ ]:
import pandas as pd
import numpy as np

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing input file: {DATA_PATH}')
if not MAP_PATH.exists():
    raise FileNotFoundError(f'Missing mapping file: {MAP_PATH}')

mapping = pd.read_csv(MAP_PATH, dtype=str).fillna('')
mapping['Variable'] = mapping['Variable'].astype(str).str.strip()
mapping['MappingBucket'] = mapping['MappingBucket'].astype(str).str.strip()
mapping['Category'] = mapping['Category'].astype(str).str.strip()
mapping['TargetIRI'] = mapping['TargetIRI'].astype(str).str.strip()
mapping['Confidence'] = mapping['Confidence'].astype(str).str.strip()

# Read full subset as strings first
raw = pd.read_csv(DATA_PATH, sep='\t', dtype=str)
print('Rows:', len(raw), 'Cols:', len(raw.columns))
print('Unique PATNO:', raw['PATNO'].astype(str).nunique() if 'PATNO' in raw.columns else 'PATNO missing')
raw.head(3)

## 3) Define core variables and semantic domains

In [ ]:
SUBJECT_META = ['PATNO', 'COHORT', 'subgroup', 'SEX', 'EDUCYRS', 'fampd_bin', 'APOE']
VISIT_META = ['EVENT_ID', 'YEAR', 'visit_date', 'age_at_visit']
TARGET_VAR = 'updrs3_score'
REQUIRED_VISITS = ['BL', 'V04', 'V06', 'V08', 'V10']

MOTOR_VARS = ['updrs1_score', 'updrs2_score', 'updrs3_score', 'updrs4_score', 'hy']
COGNITION_VARS = ['moca', 'cogstate', 'MCI_testscores']
BIOMARKER_VARS = ['abeta', 'ptau', 'asyn']
IMAGING_VARS = [
    'mia_lowput_expected',
    'MIA_CAUDATE_L', 'MIA_CAUDATE_R', 'MIA_CAUDATE_BILAT',
    'MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_PUTAMEN_BILAT',
    'MIA_STRIATUM_L', 'MIA_STRIATUM_R', 'MIA_STRIATUM_BILAT'
]
DIAGNOSIS_VARS = ['PRIMDIAG']

ALL_ANALYSIS_VARS = SUBJECT_META + VISIT_META + DIAGNOSIS_VARS + MOTOR_VARS + COGNITION_VARS + BIOMARKER_VARS + IMAGING_VARS
AVAILABLE_ANALYSIS_VARS = [c for c in ALL_ANALYSIS_VARS if c in raw.columns]
missing_for_analysis = [c for c in ALL_ANALYSIS_VARS if c not in raw.columns]

print('Available analysis vars:', len(AVAILABLE_ANALYSIS_VARS))
print('Missing analysis vars:', missing_for_analysis)
print('Required visits:', REQUIRED_VISITS)

## 4) Basic cohort validation

We require:
- subject identifier (`PATNO`)
- visit identifier (`EVENT_ID`)
- target variable (`updrs3_score`)
- longitudinal index (`YEAR`)

This notebook is designed for a fixed subset with 5 visits per subject: BL, V04, V06, V08, and V10.

In [ ]:
required_cols = ['PATNO', 'EVENT_ID', 'YEAR', TARGET_VAR]
missing_required = [c for c in required_cols if c not in raw.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

# Keep only columns needed for analysis
cohort = raw[AVAILABLE_ANALYSIS_VARS].copy()

# Clean key fields
for c in ['PATNO', 'EVENT_ID', 'YEAR', TARGET_VAR]:
    cohort[c] = cohort[c].astype(str).str.strip()

cohort = cohort[(cohort['PATNO'] != '') & (cohort['EVENT_ID'] != '')].copy()

visit_counts = cohort.groupby('PATNO')['EVENT_ID'].nunique().sort_values()
visit_counts_df = visit_counts.reset_index(name='n_visits')
print('Subjects:', cohort['PATNO'].nunique())
print('Visit count distribution:')
print(visit_counts.value_counts().sort_index())
visit_counts_df.head()

## 5) Build the trajectory cohort

For the first analysis, we retain subjects with the fixed visit design BL, V04, V06, V08, and V10, together with non-missing target values and valid longitudinal indices.

In [ ]:
def to_float_safe(x):
    s = str(x).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def normalize_event_id(x):
    return str(x).strip().upper()


cohort['YEAR_num'] = cohort['YEAR'].apply(to_float_safe)
cohort['updrs3_num'] = cohort[TARGET_VAR].apply(to_float_safe)
cohort['EVENT_ID_norm'] = cohort['EVENT_ID'].apply(normalize_event_id)

# Keep rows with valid target and valid year
traj = cohort[~cohort['YEAR_num'].isna() & ~cohort['updrs3_num'].isna()].copy()

# Restrict to required visit set
traj = traj[traj['EVENT_ID_norm'].isin(REQUIRED_VISITS)].copy()

# Keep only subjects having all required visits
visit_sets = traj.groupby('PATNO')['EVENT_ID_norm'].apply(lambda x: set(x)).reset_index(name='visit_set')
eligible_subjects = visit_sets[visit_sets['visit_set'].apply(lambda s: set(REQUIRED_VISITS).issubset(s))]['PATNO'].tolist()
traj5 = traj[traj['PATNO'].isin(eligible_subjects)].copy()

# Retain one row per PATNO / visit if duplicates exist
traj5 = traj5.sort_values(['PATNO', 'YEAR_num', 'EVENT_ID_norm']).copy()
traj5 = traj5.drop_duplicates(subset=['PATNO', 'EVENT_ID_norm'], keep='first').copy()

# Recheck completeness after duplicate removal
visit_counts_fixed = traj5.groupby('PATNO')['EVENT_ID_norm'].nunique()
valid_subjects = visit_counts_fixed[visit_counts_fixed == len(REQUIRED_VISITS)].index.tolist()
traj5 = traj5[traj5['PATNO'].isin(valid_subjects)].copy()

visit_order_map = {'BL': 1, 'V04': 2, 'V06': 3, 'V08': 4, 'V10': 5}
traj5['visit_order'] = traj5['EVENT_ID_norm'].map(visit_order_map)
traj5 = traj5.sort_values(['PATNO', 'visit_order']).copy()

print('Trajectory cohort subjects:', traj5['PATNO'].nunique())
print('Trajectory cohort rows:', len(traj5))
print('Expected rows = subjects x 5:', traj5['PATNO'].nunique() * 5)
traj5[['PATNO', 'EVENT_ID', 'EVENT_ID_norm', 'YEAR', TARGET_VAR]].head(15)

## 6) Derive subject-level progression summaries

Primary summary:
- baseline score (`BL`)
- final score (`V10`)
- absolute change
- annualised change

In [ ]:
rows = []
for patno, g in traj5.groupby('PATNO'):
    g = g.sort_values('visit_order')
    first = g[g['EVENT_ID_norm'] == 'BL'].iloc[0]
    last = g[g['EVENT_ID_norm'] == 'V10'].iloc[0]
    year_diff = float(last['YEAR_num']) - float(first['YEAR_num'])
    score_diff = float(last['updrs3_num']) - float(first['updrs3_num'])
    annualised_change = score_diff / year_diff if year_diff != 0 else np.nan
    rows.append({
        'PATNO': patno,
        'baseline_event': first['EVENT_ID_norm'],
        'last_event': last['EVENT_ID_norm'],
        'baseline_year': first['YEAR_num'],
        'last_year': last['YEAR_num'],
        'baseline_updrs3': first['updrs3_num'],
        'last_updrs3': last['updrs3_num'],
        'updrs3_change': score_diff,
        'annualised_updrs3_change': annualised_change,
        'n_visits_used': len(g),
        'baseline_hy': to_float_safe(first['hy']) if 'hy' in g.columns else np.nan,
        'baseline_moca': to_float_safe(first['moca']) if 'moca' in g.columns else np.nan,
        'baseline_primdiag': first['PRIMDIAG'] if 'PRIMDIAG' in g.columns else ''
    })

subject_progression = pd.DataFrame(rows)
subject_progression = subject_progression.sort_values('annualised_updrs3_change').reset_index(drop=True)
subject_progression.head(10)

## 7) Assign progression groups

Subjects are grouped into tertiles based on annualised change in `updrs3_score`:
- slow progression
- intermediate progression
- faster progression

In [ ]:
labels = ['slow', 'intermediate', 'faster']
subject_progression['progression_group'] = pd.qcut(
    subject_progression['annualised_updrs3_change'],
    q=3,
    labels=labels,
    duplicates='drop'
)

# Fallback in case identical values reduce quantiles
if subject_progression['progression_group'].isna().any():
    ranked = subject_progression['annualised_updrs3_change'].rank(method='first')
    subject_progression['progression_group'] = pd.qcut(ranked, q=3, labels=labels)

print(subject_progression['progression_group'].value_counts(dropna=False))
subject_progression.head(10)

## 8) Merge progression groups back to visit-level data

In [ ]:
traj5 = traj5.merge(
    subject_progression[['PATNO', 'progression_group', 'annualised_updrs3_change']],
    on='PATNO',
    how='left'
)
traj5[['PATNO', 'EVENT_ID_norm', 'YEAR', TARGET_VAR, 'progression_group']].head(15)

## 9) Cohort and progression summaries

In [ ]:
cohort_summary = pd.DataFrame([
    {'metric': 'n_subjects', 'value': int(traj5['PATNO'].nunique())},
    {'metric': 'n_rows', 'value': int(len(traj5))},
    {'metric': 'n_visits_per_subject_expected', 'value': 5},
    {'metric': 'mean_baseline_updrs3', 'value': round(subject_progression['baseline_updrs3'].mean(), 3)},
    {'metric': 'mean_last_updrs3', 'value': round(subject_progression['last_updrs3'].mean(), 3)},
    {'metric': 'mean_updrs3_change', 'value': round(subject_progression['updrs3_change'].mean(), 3)},
    {'metric': 'mean_annualised_updrs3_change', 'value': round(subject_progression['annualised_updrs3_change'].mean(), 3)}
])

progression_summary = subject_progression.groupby('progression_group').agg(
    n_subjects=('PATNO', 'nunique'),
    mean_baseline_updrs3=('baseline_updrs3', 'mean'),
    mean_last_updrs3=('last_updrs3', 'mean'),
    mean_change=('updrs3_change', 'mean'),
    mean_annualised_change=('annualised_updrs3_change', 'mean'),
    mean_baseline_hy=('baseline_hy', 'mean'),
    mean_baseline_moca=('baseline_moca', 'mean')
).reset_index()

print('--- Cohort summary ---')
print(cohort_summary)
print('\n--- Progression summary ---')
print(progression_summary)

## 10) Ontology-grounded descriptive summaries by domain

In [ ]:
def safe_mean(df, cols):
    out = {}
    for c in cols:
        if c in df.columns:
            vals = df[c].apply(to_float_safe)
            out[c] = vals.mean(skipna=True)
    return out

# Baseline rows only for domain summaries
baseline_rows = traj5[traj5['EVENT_ID_norm'] == 'BL'].copy()

baseline_domain_rows = []
for grp, g in baseline_rows.groupby('progression_group'):
    row = {'progression_group': grp, 'n_subjects': g['PATNO'].nunique()}
    row.update({f'motor__{k}': v for k, v in safe_mean(g, [c for c in MOTOR_VARS if c in g.columns]).items()})
    row.update({f'cognition__{k}': v for k, v in safe_mean(g, [c for c in COGNITION_VARS if c in g.columns]).items()})
    row.update({f'biomarker__{k}': v for k, v in safe_mean(g, [c for c in BIOMARKER_VARS if c in g.columns]).items()})
    row.update({f'imaging__{k}': v for k, v in safe_mean(g, [c for c in IMAGING_VARS if c in g.columns]).items()})
    baseline_domain_rows.append(row)

baseline_domain_summary = pd.DataFrame(baseline_domain_rows)
baseline_domain_summary

## 11) Diagnosis profile by progression group

In [ ]:
if 'PRIMDIAG' in baseline_rows.columns:
    diagnosis_profile = (
        baseline_rows.groupby(['progression_group', 'PRIMDIAG'])['PATNO']
        .nunique()
        .reset_index(name='n_subjects')
        .sort_values(['progression_group', 'n_subjects'], ascending=[True, False])
    )
else:
    diagnosis_profile = pd.DataFrame(columns=['progression_group', 'PRIMDIAG', 'n_subjects'])

diagnosis_profile.head(20)

## 12) Trajectory visualisations

In [ ]:
import matplotlib.pyplot as plt

# Figure 1: all subject trajectories coloured by group
plt.figure(figsize=(8, 5))
for grp in ['slow', 'intermediate', 'faster']:
    ggrp = traj5[traj5['progression_group'] == grp]
    for patno, g in ggrp.groupby('PATNO'):
        g = g.sort_values('visit_order')
        plt.plot(g['visit_order'], g['updrs3_num'], alpha=0.5)
plt.xticks([1, 2, 3, 4, 5], REQUIRED_VISITS)
plt.xlabel('Visit')
plt.ylabel('updrs3_score')
plt.title('Subject-level longitudinal trajectories for updrs3_score')
fig1 = FIG_DIR / 'updrs3_subject_trajectories.png'
plt.tight_layout()
plt.savefig(fig1, dpi=200)
plt.show()

# Figure 2: mean trajectory by progression group
plt.figure(figsize=(8, 5))
for grp in ['slow', 'intermediate', 'faster']:
    ggrp = traj5[traj5['progression_group'] == grp].copy()
    if len(ggrp) == 0:
        continue
    means = ggrp.groupby('visit_order')['updrs3_num'].mean().reset_index()
    plt.plot(means['visit_order'], means['updrs3_num'], marker='o', label=grp)
plt.xticks([1, 2, 3, 4, 5], REQUIRED_VISITS)
plt.xlabel('Visit')
plt.ylabel('Mean updrs3_score')
plt.title('Mean updrs3_score trajectory by progression group')
plt.legend()
fig2 = FIG_DIR / 'updrs3_group_mean_trajectories.png'
plt.tight_layout()
plt.savefig(fig2, dpi=200)
plt.show()

# Figure 3: boxplot of annualised change
plt.figure(figsize=(7, 5))
plot_data = [
    subject_progression.loc[subject_progression['progression_group'] == grp, 'annualised_updrs3_change'].dropna().values
    for grp in ['slow', 'intermediate', 'faster']
]
plt.boxplot(plot_data, labels=['slow', 'intermediate', 'faster'])
plt.ylabel('Annualised change in updrs3_score')
plt.title('Distribution of annualised motor progression by group')
fig3 = FIG_DIR / 'updrs3_annualised_change_boxplot.png'
plt.tight_layout()
plt.savefig(fig3, dpi=200)
plt.show()

print('Saved figures to:', FIG_DIR)

## 13) Export results

In [ ]:
cohort_summary.to_csv(TRAJ_OUT_DIR / 'cohort_summary.csv', index=False)
progression_summary.to_csv(TRAJ_OUT_DIR / 'progression_summary.csv', index=False)
subject_progression.to_csv(TRAJ_OUT_DIR / 'subject_progression_table.csv', index=False)
traj5.to_csv(TRAJ_OUT_DIR / 'trajectory_visit_level_table.csv', index=False)
baseline_domain_summary.to_csv(TRAJ_OUT_DIR / 'baseline_domain_summary.csv', index=False)
diagnosis_profile.to_csv(TRAJ_OUT_DIR / 'diagnosis_profile_by_group.csv', index=False)

print('Wrote:')
for fn in [
    'cohort_summary.csv',
    'progression_summary.csv',
    'subject_progression_table.csv',
    'trajectory_visit_level_table.csv',
    'baseline_domain_summary.csv',
    'diagnosis_profile_by_group.csv'
]:
    print('-', TRAJ_OUT_DIR / fn)

## 14) Quick interpretation-ready outputs

These summaries can support the first draft of the trajectory analysis results section.

In [ ]:
print('--- progression_summary ---')
print(progression_summary.round(3).to_string(index=False))

print('\n--- diagnosis_profile_by_group (top rows) ---')
print(diagnosis_profile.head(20).to_string(index=False))

print('\n--- baseline_domain_summary ---')
print(baseline_domain_summary.round(3).to_string(index=False))